# B2-024-gpu-scientific-ml-capstone — Practice p07 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** gpu-colab-l4-workflow

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import os
import tempfile

## Solution

A checkpoint must hold everything the next step depends on (Session 1, §6): the weights, Adam's state (step counts and both moments), the step counter, and the global RNG state that `torch.randint` will use to draw the next mini-batch.
`torch.load(..., weights_only=True)` accepts this dict because it contains only tensors, an `int`, `None`, and (on a GPU) a list of tensors.

In [ ]:
def save_checkpoint(path, model, optimizer, step):
    state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": int(step),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    torch.save(state, path)


def load_checkpoint(path, model, optimizer):
    state = torch.load(path, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    torch.set_rng_state(state["torch_rng"])
    if state["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    return int(state["step"])


def load_checkpoint_no_rng(path, model, optimizer):
    # Wrong variant: identical, except that the RNG state is not restored.
    state = torch.load(path, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    return int(state["step"])


x, y = capstone_data.train_rows("shapes_supervised")


def train_steps(model, optimizer, n_steps):
    losses = []
    for _ in range(n_steps):
        idx = torch.randint(0, 600, (64,))          # global generator
        optimizer.zero_grad(set_to_none=True)
        logits = model(x[idx])
        loss = F.cross_entropy(logits, y[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.detach())
    return losses


def fresh_run():
    torch.manual_seed(SEED)
    model = capstone_data.ShapeCNN()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    return model, optimizer

**Probes.** The uninterrupted run gives `L21`.
The resumed run saves at step 20, starts a "new session" (fresh model and optimizer, then the generator is moved elsewhere with `SEED + 1`), loads, and takes one more step.

In [ ]:
# Probe 1: uninterrupted
model, optimizer = fresh_run()
L21 = train_steps(model, optimizer, 21)[20]

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "ckpt.pt")

    # Probe 2: resumed
    model, optimizer = fresh_run()
    train_steps(model, optimizer, 20)
    save_checkpoint(path, model, optimizer, 20)
    saved_state = {k: v.clone() for k, v in model.state_dict().items()}

    model2, optimizer2 = fresh_run()
    torch.manual_seed(SEED + 1)
    k = load_checkpoint(path, model2, optimizer2)
    state_after_load_equal = all(torch.equal(model2.state_dict()[name], saved_state[name]) for name in saved_state)
    resumed_loss = train_steps(model2, optimizer2, 1)[0]

    # Probe 3: wrong variant (no RNG restore)
    model3, optimizer3 = fresh_run()
    torch.manual_seed(SEED + 1)
    k_wrong = load_checkpoint_no_rng(path, model3, optimizer3)
    wrong_loss = train_steps(model3, optimizer3, 1)[0]

    # Probe 4: file contents
    stored = torch.load(path, weights_only=True)
    stored_keys = set(stored)
    stored_cuda_rng = stored["cuda_rng"]
    stored_step_type = type(stored["step"])

print("L21", L21.item(), "| resumed", resumed_loss.item(), "| no-RNG", wrong_loss.item())
print("k =", k, "| keys:", sorted(stored_keys), "| cuda_rng:", stored_cuda_rng)

**Why the wrong variant differs, and why it still trains normally.**
Without the RNG restore the global generator is in the state set by `torch.manual_seed(SEED + 1)`, so `torch.randint` draws a different mini-batch for step 21 than the uninterrupted run drew, and the loss on that different batch differs.
The weights and Adam state were restored correctly, so the update itself is a perfectly valid Adam step on a valid batch: training continues normally, but it is no longer the same experiment, which only the exact-equality certificate exposes.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, the same `save_checkpoint` / `load_checkpoint` protect a long run against disconnects. Use a replacement config for the same training code (move each batch with `.to(device)`), and write checkpoints to mounted persistent storage with an atomic rename (`torch.save` to `path + ".tmp"`, then `os.replace`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "batch_size": 512, "steps": 20000,
             "checkpoint_every": 2000, "checkpoint_dir": "/content/drive/MyDrive/b2-024-checkpoints"}
```

**What to observe.** `"cuda_rng"` is now a non-empty list; a resume after a forced runtime restart continues from the last saved step; with CUDA kernels the next-step loss may differ from the uninterrupted run in the last bits unless deterministic algorithms are enabled, so compare with a tolerance there and keep `torch.equal` for the CPU certificate.

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
# Probe 2
assert k == 20
assert torch.equal(resumed_loss, L21)
assert state_after_load_equal
# Probe 3
assert k_wrong == 20
assert not torch.equal(wrong_loss, L21)
assert torch.isfinite(wrong_loss)
# Probe 4
assert stored_keys == {"model", "optimizer", "step", "torch_rng", "cuda_rng"}
assert stored_step_type is int
if not torch.cuda.is_available():
    assert stored_cuda_rng is None